<a href="https://colab.research.google.com/github/julialhanson/BTTAI-Work/blob/main/LegalDuel1A.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
!pip install PyPDF2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 3.5 MB/s eta 0:00:00


In [36]:
# importing libraries here

from transformers import BartTokenizer, BartForConditionalGeneration
import torch
from PyPDF2 import PdfReader
from google.colab import files
import spacy

nlp = spacy.load("en_core_web_sm")

In [10]:
# this is how we are able to upload the different chronologies into the document
# run this cell to add new docs

uploaded = files.upload()

Saving ChronologyTest.pdf to ChronologyTest.pdf


In [12]:
# looking at our input file here, only used for looking at document, not as input
# for model

filepath = "ChronologyTest.pdf"
reader = PdfReader(filepath)

numPages = len(reader.pages)

for pageNum in range(numPages):
  page = reader.pages[pageNum]
  text = page.extract_text()
  print("Page:" + str(pageNum + 1))
  print(text)

Page:1
Attorney:
Mr.
Hardwick,
can
you
please
state
your
full
name
for
the
record?
John
Hardwick:
Johnathan
Hardwick.
Attorney:
Mr.
Hardwick,
when
did
the
hit-and-run
incident
take
place?
John
Hardwick:
It
was,
uh,
around
July
2019.
Or
maybe
August.
No,
it
was
definitely
July
2019.
Attorney:
And
where
did
the
incident
occur?
John
Hardwick:
It
happened
on
Greenway
Avenue,
near
the
intersection
with
Birch
Street.
I
was
just
walking
home
from
work.
Attorney:
How
long
had
you
been
walking
before
the
incident?
John
Hardwick:
I
had
been
walking
for,
oh,
maybe
10
minutes.
It
was
part
of
my
usual
route,
something
I’d
done
for
years.
Attorney:
Did
you
notice
anything
unusual
before
the
incident?
John
Hardwick:
Not
really.
I
did
hear
a
car
revving
up,
but
I
didn’t
think
much
of
it
until
it
got
really
close.
Attorney:
Can
you
describe
what
happened
when
the
car
hit
you?
John
Hardwick:
The
car
came
up
fast
behind
me.
I
didn’t
even
see
it
coming.
Next
thing
I
know,
I
was
on
the
ground.
It
hit
my
le

In [34]:
# clean text function

def clean_text(text):
  nlpText = nlp(text)

  cleaned_text = []

  for word in nlpText:
    if not word.is_stop and not word.is_punct:
      cleaned_text.append(word.lemma_.lower())

  return " ".join(cleaned_text)


In [30]:
# extracting text for the model

def extract_text_from_pdf(filepath):
  reader = PdfReader(filepath)
  text = ""

  for i in range(len(reader.pages)):
    page = reader.pages[i]
    pageText = page.extract_text()

    pageText = pageText.replace("\n", " ").strip()
    text += pageText + "\n\n"

    final_text = clean_text(text)
  return text

In [32]:
# summarzing text function

def summarize_transcript(text, max_length=1024):
  inputs = tokenizer(text, max_length=max_length, return_tensors="pt", truncation=True)
  summary_ids = model.generate(inputs['input_ids'], max_length=150, min_length=40, length_penalty=2.0, num_beams=4, early_stopping=True)
  summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
  return summary

In [38]:
tokenizer = BartTokenizer.from_pretrained("facebook/bart-large-cnn")
model = BartForConditionalGeneration.from_pretrained("facebook/bart-large-cnn")

def create_chronology(filepath):
  transcript = extract_text_from_pdf(filepath)
  summary = summarize_text(transcript)
  return summary

chronology = create_chronology(filepath)
print(chronology)

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

Johnathan Hardwick says he was walking home from work when he was hit by a car. He says he didn't see the car coming and didn't get a good look at the vehicle. He sustained injuries to his leg and hip, but no broken bones.
